In [29]:
import re, copy
import requests
import json 

In [30]:
# Function to get game data
API_KEY = "C45399540985DC199A1C374C8EF89F6F"
GENRES_COUNT = 3
CATEGORIES_COUNT = 5

def GetGameInfo(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us"
    data = requests.get(url).json()
    game = data[str(appid)]["data"]
    result = {}
    result["name"] = game["name"]
    result["publisher"] = game["publishers"][0]
    result["genres"] = game["genres"]
    result["platforms"] = game["platforms"]
    result["categories"] = game["categories"]
    result["year"] = game["release_date"]["date"][-4:]
    price_overview = game.get("price_overview")
    if price_overview:
        result["price"] = game["price_overview"]["final"]
    else:
        result["price"] = 0
    return result

In [47]:
# Get Data
METADATA = {}
METADATA[1306630] = GetGameInfo(1306630)
METADATA[537110] = GetGameInfo(537110)
METADATA[2230650] = GetGameInfo(2230650)
METADATA[3887700] = GetGameInfo(3887700)
METADATA[648100] = GetGameInfo(648100)
METADATA[730] = GetGameInfo(730)
METADATA[1808500] = GetGameInfo(1808500)
METADATA[1174180] = GetGameInfo(1174180)
METADATA[2807960] = GetGameInfo(2807960)
METADATA[3809440] = GetGameInfo(3809440)
print(METADATA)

{1306630: {'name': 'Lost Ruins', 'publisher': 'DANGEN Entertainment', 'genres': [{'id': '1', 'description': 'Action'}, {'id': '25', 'description': 'Adventure'}, {'id': '23', 'description': 'Indie'}], 'platforms': {'windows': True, 'mac': True, 'linux': True}, 'categories': [{'id': 2, 'description': 'Single-player'}, {'id': 22, 'description': 'Steam Achievements'}, {'id': 28, 'description': 'Full controller support'}, {'id': 55, 'description': 'DualShock Controller Support'}, {'id': 56, 'description': 'DualShock Controller Support'}, {'id': 23, 'description': 'Steam Cloud'}, {'id': 41, 'description': 'Remote Play on Phone'}, {'id': 42, 'description': 'Remote Play on Tablet'}, {'id': 62, 'description': 'Family Sharing'}], 'year': '2021', 'price': 1999}, 537110: {'name': 'Angels of Death', 'publisher': 'Vaka Game Magazine', 'genres': [{'id': '25', 'description': 'Adventure'}, {'id': '23', 'description': 'Indie'}, {'id': '3', 'description': 'RPG'}], 'platforms': {'windows': True, 'mac': Fa

In [48]:
# Setup for a query
QUERY_SAMPLE = [
    {"ID":1306630, "Hours":11.8, "Installed":False},
    {"ID":537110, "Hours":6.6, "Installed":False},
    {"ID":2230650, "Hours":93.6, "Installed":True},
    {"ID":3887700, "Hours":152, "Installed":False},
    {"ID":648100, "Hours":35.4, "Installed":False},
]

# Baseline test
total_hours = 0.0
genre_tags = {}
for query in QUERY_SAMPLE:
    total_hours += query["Hours"]
    genre = METADATA[query["ID"]]["genres"]
    for g in genre:
        genre_tags[g["id"]] = 0.0
for query in QUERY_SAMPLE:
    genre = METADATA[query["ID"]]["genres"]
    for g in genre:
        genre_tags[g["id"]] += query["Hours"] / len(genre) / total_hours
print(genre_tags)
scores = {}
for id in METADATA:
    genre = METADATA[id]["genres"]
    scores[METADATA[id]["name"]] = 0.0
    for g in genre:
        if g["id"] in genre_tags:
            scores[METADATA[id]["name"]] += genre_tags[g["id"]]
print(scores)

{'1': 0.09129369850812737, '25': 0.15775996437319084, '23': 0.15775996437319084, '3': 0.17657537296815853, '4': 0.0781563126252505, '2': 0.16922734357604097, '37': 0.16922734357604097}
{'Lost Ruins': 0.40681362725450904, 'Angels of Death': 0.4920953017145402, 'TEVI': 0.48496993987975956, "Girls' Frontline": 0.5150300601202404, 'Raging Loop': 0.3155199287463817, 'Counter-Strike 2': 0.2605210420841683, 'ARC Raiders': 0.09129369850812737, 'Red Dead Redemption 2': 0.2490536628813182, 'Battlefield™ 6': 0.09129369850812737, 'Cheese Rolling': 0.4964373190826097}
